# Create `Splice-Junction-to-RBP-Number-of-Peaks` Dictionary

## Purpose: 

* Take `bedtools closest` output and clean it up. 
* Create dictionary of splice junction to number of times RBP binds. 
* Output dictionary. 

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, pickle, gzip

## Literals

In [2]:
cell_lines = ["K562", "HepG2"]
thresholds = [50, 100, 500, 1000, 2000, 5000, 10000]

## Load and Clean `BedTools Closest` Output

In [3]:
eclip_to_splice_junctions = {}

for cell_line in cell_lines:
    file = glob.glob("../output/peaks_to_splice_junctions/{}*.bed".format(cell_line))
    assert len(file)==1
    
    tmp_mappings = pd.read_csv(file[0], sep="\t", header=None)
    
    "Before filtering out missing mappings: {}".format(tmp_mappings.index.size)
    
    # get the distance column (last column in bed file
    distance_column = tmp_mappings.columns[-1]
    # remove the peaks that did not have any splice junctions to map to    
    tmp_mappings = tmp_mappings[tmp_mappings[distance_column]>=0]
    
    "After filtering out missing mappings: {}".format(tmp_mappings.index.size)
        
    # filter for the RBP, unique splice junction ID, and distance columns
    tmp_mappings = tmp_mappings[[3, 9, distance_column]]
    
    # parse out RBP column
    tmp_mappings["RBP"] = tmp_mappings[3].str.split("_").str[0]
    # change column names for clarity
    tmp_mappings = tmp_mappings.rename(
        columns={
            9: "Splice Junction ID", 
            12: "Distance"
        }
    )
    
    # ensure there are no null values in matrix 
    assert tmp_mappings.notnull().all().all()
    
    tmp_mappings.head()  
    tmp_mappings["Distance"].min()
    
    # save the refined dataset
    eclip_to_splice_junctions[cell_line] = tmp_mappings
    


'Before filtering out missing mappings: 358745'

'After filtering out missing mappings: 357899'

,3,Splice Junction ID,Distance,RBP
0,AQR_K562_IDR,chr1_15947_-,0,AQR
1,BUD13_K562_IDR,chr1_16606_-,276,BUD13
2,AQR_K562_IDR,chr1_16606_-,288,AQR
3,AQR_K562_IDR,chr1_16606_-,97,AQR
4,CSTF2T_K562_IDR,chr1_17605_-,33,CSTF2T


0

'Before filtering out missing mappings: 415279'

'After filtering out missing mappings: 414621'

,3,Splice Junction ID,Distance,RBP
0,PRPF4_HepG2_IDR,chr1_17605_-,39,PRPF4
1,EFTUD2_HepG2_IDR,chr1_17605_-,56,EFTUD2
2,PRPF8_HepG2_IDR,chr1_187128_-,0,PRPF8
3,PPIG_HepG2_IDR,chr1_187577_-,13,PPIG
4,TARDBP_HepG2_IDR,chr1_594756_-,2251,TARDBP


0

## Get Dictionary of Splice Junction to RBPs Bound


**Dictionary structure**: 

```
{
    cell_line: {
        distance threshold:{
            splice junction id: {
                RBP name: # eCLIP Peaks 
            }
        }
    }
}
```

In [4]:
# dict following paradigm described above 
all_splice_junction_to_rbp_num_peaks = {}

# for each cell line 
for cell_line in cell_lines:
    
    # get all eCLIP to splice junction mappings for the cell line 
    cell_line_mappings = eclip_to_splice_junctions[cell_line]
    
    all_splice_junction_to_rbp_num_peaks[cell_line] = {}
    
    # for each distance threshold 
    for threshold in thresholds: 
        
        # subset to less than distance 
        # THIS ASSUMES THAT ALL DISTANCES ARE POSITIVE
        threshold_mappings = cell_line_mappings[cell_line_mappings["Distance"]<=threshold]
        
        # get a count for the number of peaks of a given RBP at a specific splice junction for a distance threshold 
        # use pd.unstack method to convert output series into a 2D matrix where null values are made into 0
        threshold_mappings = threshold_mappings.groupby(["Splice Junction ID", "RBP"]).size().unstack(fill_value=0)
        
        # get the rbps that should be included in the dataset but had no data show up 
        # (we need to indicate that these data are available but are all "0" values)
        missing_rbps = set(cell_line_mappings["RBP"].unique().tolist()) - set(threshold_mappings.columns.to_list())
            
        # if there are missing rbps 
        if len(missing_rbps)>0: 
            # fill all values for the rbp as 0 
            for rbp in missing_rbps: 
                threshold_mappings[rbp] = 0
        
        # sort RBP columns by name 
        threshold_mappings = threshold_mappings.sort_index(axis=1)
        
        # convert datafraem to dict whereby first key is splice junciton id and value is another dict where 
        # sub-dict key is RBP name and value is number of peaks
        all_splice_junction_to_rbp_num_peaks[cell_line][threshold] = threshold_mappings.to_dict(orient="index")

## Output Dictionary as `GZIP Pickle` File

In [5]:
with gzip.GzipFile("../output/splice_junction_rbp_num_peaks/all_RBP_peaks_num_per_splice_junction.pkl.gz", 'wb') as file: 
    pickle.dump(all_splice_junction_to_rbp_num_peaks, file)